<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

The official checkpoints are ONNX. Loading them does not require the `onnx` package or a runtime. A graph is a protobuf message: nodes in exporter order, initializers, and the input and output shapes. `weights` walks that node list.

## Varints

Protobuf integers are base-128, little-endian, with the high bit set on every byte but the last.

## Messages

One message is a sequence of `(field_number, wire_type, payload)`. Wire 0 is a varint, 1 is 8 bytes, 2 is a length-delimited blob, and 5 is 4 bytes.

In [6]:
#| echo: false
#| output: asis
show_doc(fields)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/onnx.py#L46){target="_blank" style="float:right; font-size:smaller"}

### fields

```python
def fields(
    buf:bytes
)->list[tuple[int, int, bytes | int]]:
```

*Return `(field_number, wire_type, payload)` for one message.*

150 is the protobuf specimen for a varint: the bytes `96 01`. Field 1, wire 0, is the key byte `08` in front of that. Zigzag maps even numbers to non-negative integers and odd numbers to negative ones.

In [11]:
from fastcore.test import test_eq

test_eq(_varint(bytes([0x96, 0x01]), 0), (150, 2))
test_eq(fields(bytes([0x08, 0x96, 0x01])), [(1, 0, 150)])
test_eq([_zigzag(v) for v in range(4)], [0, -1, 1, -2])
fields(bytes([0x08, 0x96, 0x01]))

[(1, 0, 150)]

## Tensors

An initializer is a name, a dtype tag, a shape, and either raw little-endian bytes or a packed numeric field. bfloat16 is stored as the top 16 bits of a float32.

In [13]:
#| echo: false
#| output: asis
show_doc(Tensor)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/onnx.py#L106){target="_blank" style="float:right; font-size:smaller"}

### Tensor

```python
def Tensor(
    name:str, array:np.ndarray
)->None:
```

*Named initializer array.*

In [15]:
#| echo: false
#| output: asis
show_doc(parse_tensor)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/onnx.py#L128){target="_blank" style="float:right; font-size:smaller"}

### parse_tensor

```python
def parse_tensor(
    buf:bytes
)->Tensor:
```

## Attributes and nodes

In [17]:
#| echo: false
#| output: asis
show_doc(Attribute)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/onnx.py#L180){target="_blank" style="float:right; font-size:smaller"}

### Attribute

```python
def Attribute()->None:
```

*One node attribute: a scalar, bytes, or a list of those.*

In [19]:
#| echo: false
#| output: asis
show_doc(parse_attribute)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/onnx.py#L194){target="_blank" style="float:right; font-size:smaller"}

### parse_attribute

```python
def parse_attribute(
    buf:bytes
)->Attribute:
```

In [21]:
#| echo: false
#| output: asis
show_doc(Node)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/onnx.py#L219){target="_blank" style="float:right; font-size:smaller"}

### Node

```python
def Node()->None:
```

*Op type, name, input and output names, and attributes.*

In [23]:
#| echo: false
#| output: asis
show_doc(parse_node)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/onnx.py#L231){target="_blank" style="float:right; font-size:smaller"}

### parse_node

```python
def parse_node(
    buf:bytes
)->Node:
```

A node records op type, name, input and output names, and attributes. The weight loader only needs the op type and the input names, because those names are the keys of the initializers.

## The graph

In [25]:
#| echo: false
#| output: asis
show_doc(ValueInfo)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/onnx.py#L249){target="_blank" style="float:right; font-size:smaller"}

### ValueInfo

```python
def ValueInfo(
    name:str, shape:list[int | str], dtype:int
)->None:
```

*Graph input or output: name, shape, and a dtype tag.*

In [27]:
#| echo: false
#| output: asis
show_doc(parse_value_info)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/onnx.py#L259){target="_blank" style="float:right; font-size:smaller"}

### parse_value_info

```python
def parse_value_info(
    buf:bytes
)->ValueInfo:
```

In [29]:
#| echo: false
#| output: asis
show_doc(Graph)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/onnx.py#L286){target="_blank" style="float:right; font-size:smaller"}

### Graph

```python
def Graph()->None:
```

*Nodes, initializers, and the graph inputs and outputs.*

In [31]:
#| echo: false
#| output: asis
show_doc(parse_graph)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/onnx.py#L297){target="_blank" style="float:right; font-size:smaller"}

### parse_graph

```python
def parse_graph(
    buf:bytes
)->Graph:
```

In [33]:
#| echo: false
#| output: asis
show_doc(load)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/onnx.py#L315){target="_blank" style="float:right; font-size:smaller"}

### load

```python
def load(
    path:str | Path
)->Graph:
```

An ONNX file is a `ModelProto`. Field 7 is the graph. Everything the loader needs is in that message: nodes, then initializers, then inputs and outputs.